# Energy Complete Experiment

Runs the full 20-seed energy CMDL, Plain LSTM, Grouped ARDL, and ablation suite, then saves table artifacts only.

In [1]:
from argparse import Namespace
from pathlib import Path
import os
import shutil
import sys

import pandas as pd
from IPython.display import display

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

current = Path.cwd().resolve()
repo_root = next((p for p in [current, *current.parents] if (p / "experiments").exists() and (p / "config").exists()), None)
if repo_root is None:
    raise RuntimeError(f"Could not locate repo root from {current}")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from evaluation.energy_comparison import build_energy_comparison, build_mechanism_result_log
from evaluation.stratified_kstar import aggregate_per_method, build_energy_stratifiers, collect_seed_dirs, evaluate_method
from experiments import run_energy, run_energy_ablation, run_energy_ganet_baseline, run_energy_grouped_ardl, run_energy_lstm_baseline, run_energy_tft_baseline
from experiments.run_complete_20seed_suite import REALDATA_VARIANTS, cleanup, energy_common_args

PLAN_NAME = "complete_20seed_20260426"
SEEDS = list(range(20))
FORCE = False
RUN_CMDL = True
RUN_BASELINE = True
RUN_TFT = True
RUN_GANET = True
RUN_GROUPED_ARDL = True
RUN_ABLATIONS = True
N_PERM = 2000

OUTPUT_ROOT = repo_root / "outputs" / "notebook_energy" / PLAN_NAME
CMDL_DIR = OUTPUT_ROOT / "cmdl"
BASELINE_DIR = OUTPUT_ROOT / "plain_lstm"
TFT_DIR = OUTPUT_ROOT / "tft"
GANET_DIR = OUTPUT_ROOT / "ganet"
GROUPED_DIR = OUTPUT_ROOT / "grouped_ardl"
ABLATION_DIR = OUTPUT_ROOT / "ablation"
COMPARISON_DIR = OUTPUT_ROOT / "comparison"
for path in [CMDL_DIR, BASELINE_DIR, TFT_DIR, GANET_DIR, GROUPED_DIR, ABLATION_DIR, COMPARISON_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def summary_exists(run_dir: Path) -> bool:
    return (run_dir / "summary.json").exists()

def run_task(label: str, run_dir: Path, callback) -> None:
    if summary_exists(run_dir) and not FORCE:
        print(f"[skip] {label}: {run_dir}")
        return
    if run_dir.exists() and (FORCE or not summary_exists(run_dir)):
        reason = "force rerun" if FORCE else "incomplete artifact"
        print(f"[clean] {reason}: {run_dir}")
        shutil.rmtree(run_dir)
    print(f"[run] {label}: {run_dir}")
    callback()
    cleanup()
    if not summary_exists(run_dir):
        raise RuntimeError(f"Expected summary.json was not created for {label}: {run_dir}")

settings = pd.Series({"plan_name": PLAN_NAME, "seeds": SEEDS, "force": FORCE, "n_perm": N_PERM, "output_root": OUTPUT_ROOT}).to_frame("value")
display(settings)

c:\Users\42155\anaconda3\envs\PTenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,value
plan_name,complete_20seed_20260426
seeds,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
force,False
n_perm,2000
output_root,C:\DevSpace\PyDevspace\CMDL\outputs\notebook_e...


In [2]:
if RUN_CMDL:
    for seed in SEEDS:
        name = f"energy_cmdl_seed{seed}"
        args = Namespace(**energy_common_args(CMDL_DIR), seed=seed, seeds=None, experiment_name=name)
        run_task(f"energy CMDL seed {seed}", CMDL_DIR / name, lambda args=args: run_energy.run_experiment(args))

if RUN_BASELINE:
    for seed in SEEDS:
        name = f"energy_lstm_seed{seed}"
        args = Namespace(**energy_common_args(BASELINE_DIR), seed=seed, seeds=None, experiment_name=name)
        run_task(f"energy Plain LSTM seed {seed}", BASELINE_DIR / name, lambda args=args: run_energy_lstm_baseline.run_experiment(args))

if RUN_GROUPED_ARDL:
    for seed in SEEDS:
        name = f"energy_grouped_ardl_seed{seed}"
        args = Namespace(**energy_common_args(GROUPED_DIR), seed=seed, seeds=None, experiment_name=name)
        run_task(f"energy Grouped ARDL seed {seed}", GROUPED_DIR / name, lambda args=args: run_energy_grouped_ardl.run_experiment(args))

if RUN_ABLATIONS:
    ablation_args = Namespace(**energy_common_args(ABLATION_DIR), variant="all", seeds=SEEDS, experiment_prefix="energy_ablation")
    for seed in SEEDS:
        for variant in REALDATA_VARIANTS:
            name = f"energy_ablation_{variant}_seed{seed}"
            run_task(f"energy ablation {variant} seed {seed}", ABLATION_DIR / name, lambda variant=variant, seed=seed: run_energy_ablation.run_variant(ablation_args, variant, seed))

[skip] energy CMDL seed 0: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_energy\complete_20seed_20260426\cmdl\energy_cmdl_seed0
[skip] energy CMDL seed 1: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_energy\complete_20seed_20260426\cmdl\energy_cmdl_seed1
[skip] energy CMDL seed 2: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_energy\complete_20seed_20260426\cmdl\energy_cmdl_seed2
[skip] energy CMDL seed 3: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_energy\complete_20seed_20260426\cmdl\energy_cmdl_seed3
[skip] energy CMDL seed 4: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_energy\complete_20seed_20260426\cmdl\energy_cmdl_seed4
[skip] energy CMDL seed 5: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_energy\complete_20seed_20260426\cmdl\energy_cmdl_seed5
[skip] energy CMDL seed 6: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_energy\complete_20seed_20260426\cmdl\energy_cmdl_seed6
[skip] energy CMDL seed 7: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_energy\complete_20seed_20260426\cmdl\en

In [3]:
if RUN_TFT:
    for seed in SEEDS:
        name = f"energy_tft_seed{seed}"
        args = Namespace(**energy_common_args(TFT_DIR), seed=seed, seeds=None, experiment_name=name)
        run_task(f"energy TFT seed {seed}", TFT_DIR / name, lambda args=args: run_energy_tft_baseline.run_experiment(args))

if RUN_GANET:
    for seed in SEEDS:
        name = f"energy_ganet_seed{seed}"
        args = Namespace(**energy_common_args(GANET_DIR), seed=seed, seeds=None, experiment_name=name)
        run_task(f"energy GA-Net seed {seed}", GANET_DIR / name, lambda args=args: run_energy_ganet_baseline.run_experiment(args))

[run] energy TFT seed 0: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_energy\complete_20seed_20260426\tft\energy_tft_seed0
[energy_tft_seed0] epoch=001 train_task=1.2818 val_task=17.2270 val_mae=2.5523 val_r2=-0.0474
[energy_tft_seed0] epoch=010 train_task=1.1065 val_task=15.9088 val_mae=2.3573 val_r2=0.0469
[energy_tft_seed0] epoch=020 train_task=0.7317 val_task=15.3149 val_mae=2.2247 val_r2=0.0738
[energy_tft_seed0] epoch=030 train_task=0.5093 val_task=15.1443 val_mae=2.2381 val_r2=0.0807
[energy_tft_seed0] epoch=040 train_task=0.3593 val_task=15.2310 val_mae=2.2582 val_r2=0.0567
[energy_tft_seed0] epoch=050 train_task=0.2826 val_task=15.6092 val_mae=2.2763 val_r2=0.0272
[energy_tft_seed0] early stopping at epoch 54
[run] energy TFT seed 1: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_energy\complete_20seed_20260426\tft\energy_tft_seed1
[energy_tft_seed1] epoch=001 train_task=2.3595 val_task=18.9007 val_mae=2.7566 val_r2=-0.1579
[energy_tft_seed1] epoch=010 train_task=1.1465 val_task

In [4]:
comparison = build_energy_comparison(
    cmdl_root=CMDL_DIR,
    baseline_root=BASELINE_DIR,
    tft_root=TFT_DIR,
    ganet_root=GANET_DIR,
    ablation_root=ABLATION_DIR,
    grouped_ardl_root=GROUPED_DIR,
)
mechanism_log = build_mechanism_result_log(comparison)

raw_csv = repo_root / "data" / "energy" / "raw" / "energy_wgi_merged.csv"
stratifiers = build_energy_stratifiers(raw_csv)
method_dirs = {
    "CMDL": collect_seed_dirs(CMDL_DIR, "energy_cmdl_"),
    "Plain LSTM": collect_seed_dirs(BASELINE_DIR, "energy_lstm_"),
    "TFT": collect_seed_dirs(TFT_DIR, "energy_tft_"),
    "GA-Net": collect_seed_dirs(GANET_DIR, "energy_ganet_"),
    "No Recon Regularization": collect_seed_dirs(ABLATION_DIR, "energy_ablation_no_recon_regularization_"),
    "No AC Encoder": collect_seed_dirs(ABLATION_DIR, "energy_ablation_no_ac_encoder_"),
    "Uniform Lag": collect_seed_dirs(ABLATION_DIR, "energy_ablation_uniform_lag_"),
}
per_seed_frames = [evaluate_method(method, dirs, stratifiers, n_perm=N_PERM) for method, dirs in method_dirs.items() if dirs]
stratified_per_seed = pd.concat(per_seed_frames, ignore_index=True, sort=False) if per_seed_frames else pd.DataFrame()
stratified_aggregated = aggregate_per_method(stratified_per_seed) if not stratified_per_seed.empty else pd.DataFrame()

comparison.to_csv(COMPARISON_DIR / "energy_comparison.csv", index=False)
mechanism_log.to_csv(COMPARISON_DIR / "energy_mechanism_result_log.csv", index=False)
stratified_per_seed.to_csv(COMPARISON_DIR / "energy_stratified_kstar_per_seed.csv", index=False)
stratified_aggregated.to_csv(COMPARISON_DIR / "energy_stratified_kstar_aggregated.csv", index=False)

tables = {
    "energy_comparison": comparison,
    "energy_mechanism_result_log": mechanism_log,
    "energy_stratified_kstar_per_seed": stratified_per_seed,
    "energy_stratified_kstar_aggregated": stratified_aggregated,
}
pd.Series({name: len(frame) for name, frame in tables.items()}, name="rows").to_frame()

,rows
energy_comparison,160
energy_mechanism_result_log,6
energy_stratified_kstar_per_seed,420
energy_stratified_kstar_aggregated,21


In [5]:
for name, frame in tables.items():
    print(f"\n=== {name} ({len(frame)} rows) ===")
    display(frame.head(20))


=== energy_comparison (160 rows) ===


,family,display_name,experiment,model,variant,lag_method,tracking_backend,device,domain,scenario,...,test_effective_lag_mean,test_grouped_ardl_best_lag_mean,test_grouped_ardl_effective_lag_mean,test_grouped_ardl_group_count,test_grouped_ardl_low_best_lag,test_grouped_ardl_low_effective_lag,test_grouped_ardl_mid_best_lag,test_grouped_ardl_mid_effective_lag,test_grouped_ardl_high_best_lag,test_grouped_ardl_high_effective_lag
0,ablation,No AC Encoder,energy_ablation_no_ac_encoder_seed0,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,energy,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ablation,No AC Encoder,energy_ablation_no_ac_encoder_seed1,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,energy,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ablation,No AC Encoder,energy_ablation_no_ac_encoder_seed2,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,energy,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ablation,No AC Encoder,energy_ablation_no_ac_encoder_seed3,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,energy,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ablation,No AC Encoder,energy_ablation_no_ac_encoder_seed4,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,energy,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ablation,No AC Encoder,energy_ablation_no_ac_encoder_seed5,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,energy,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ablation,No AC Encoder,energy_ablation_no_ac_encoder_seed6,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,energy,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ablation,No AC Encoder,energy_ablation_no_ac_encoder_seed7,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,energy,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ablation,No AC Encoder,energy_ablation_no_ac_encoder_seed8,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,energy,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ablation,No AC Encoder,energy_ablation_no_ac_encoder_seed9,cmdl_ablation,no_ac_encoder,learned_omega,json,cuda,energy,linear,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== energy_mechanism_result_log (6 rows) ===


,layer,question,answer,evidence
0,forecast_calibration,Does CMDL beat the matched LSTM?,yes,"mean CMDL test_r2=-0.028567153527486667, mean ..."
1,simple_baseline_calibration,Is CMDL above the simple calibrated baselines?,no,"delta_vs_persistence=-0.7874062086548632, delt..."
2,ac_gate_mechanism,Does the anchor-adjusted lag-proxy direction s...,partial,"CMDL mean adjusted rho=-0.014338563591048481, ..."
3,ac_gate_per_proxy,Are all named proxy adjusted correlations alig...,no,"candidate_positive_proxies=0/3, min_adjusted_r..."
4,ac_gate_heterogeneity,Is the learned lag gate non-degenerate?,yes,"lag_gate_sensitivity_range=0.2690467834472656,..."
5,ablation_guard,Do degenerate controls expose the heterogeneit...,yes,"No AC kstar_std=0.0, Uniform Lag top1_share=1.0."



=== energy_stratified_kstar_per_seed (420 rows) ===


,method,seed,stratifier,n_entities,spearman_rho,perm_p_two_sided,kstar_std,degenerate
0,CMDL,0,log_gdp_per_capita_train,78,-0.591017,0.0000,0.062238,False
1,CMDL,0,government_effectiveness_train,78,-0.644330,0.0000,0.062238,False
2,CMDL,0,rule_of_law_train,78,-0.680117,0.0000,0.062238,False
3,CMDL,1,log_gdp_per_capita_train,78,0.724908,0.0000,0.112889,False
4,CMDL,1,government_effectiveness_train,78,0.871799,0.0000,0.112889,False
5,CMDL,1,rule_of_law_train,78,0.891577,0.0000,0.112889,False
6,CMDL,2,log_gdp_per_capita_train,78,-0.740209,0.0000,0.144515,False
7,CMDL,2,government_effectiveness_train,78,-0.870560,0.0000,0.144515,False
8,CMDL,2,rule_of_law_train,78,-0.919397,0.0000,0.144515,False
9,CMDL,3,log_gdp_per_capita_train,78,-0.649418,0.0000,0.002025,False



=== energy_stratified_kstar_aggregated (21 rows) ===


,method,stratifier,n_seeds_total,n_seeds_valid,rho_mean,rho_median,abs_rho_mean,share_seeds_p_lt_05,share_seeds_p_lt_01,fisher_combined_p
0,CMDL,government_effectiveness_train,20,20,0.014916,0.129111,0.715528,0.90,0.85,9.034017e-77
1,CMDL,log_gdp_per_capita_train,20,20,0.011460,0.185157,0.608751,0.90,0.85,1.452259e-77
2,CMDL,rule_of_law_train,20,20,0.021033,0.236864,0.734692,0.95,0.90,1.788747e-79
3,GA-Net,government_effectiveness_train,20,0,NaN,NaN,NaN,NaN,NaN,NaN
4,GA-Net,log_gdp_per_capita_train,20,0,NaN,NaN,NaN,NaN,NaN,NaN
5,GA-Net,rule_of_law_train,20,0,NaN,NaN,NaN,NaN,NaN,NaN
6,No AC Encoder,government_effectiveness_train,20,0,NaN,NaN,NaN,NaN,NaN,NaN
7,No AC Encoder,log_gdp_per_capita_train,20,0,NaN,NaN,NaN,NaN,NaN,NaN
8,No AC Encoder,rule_of_law_train,20,0,NaN,NaN,NaN,NaN,NaN,NaN
9,No Recon Regularization,government_effectiveness_train,20,20,0.014953,0.129111,0.715554,0.90,0.85,9.034017e-77
